## tl;dr

Across `6,465` valid full-window paired states and `1,403` valid states in the registered 120–180 second interval, the frozen two-causal-tick residual rule rejected **zero** observations. The midpoint clause also rejected zero, and the microprice clause added zero rejections beyond midpoint. In this 24-market label-free capture, residual magnitude therefore supplied no observed selectivity beyond paired-top validity. All unavailable observations failed the valid-top check; none failed for staleness or a missing snapshot. This falsifies an independent residual-magnitude interpretation for the observed population, but it does not measure outcomes or authorize changing the sealed rule.


## Context & Methods

This is a **label-free mechanism diagnostic**, not a backtest. It asks whether the registered
`max(abs(midpoint residual), abs(microprice residual)) <= 2 * causal tick` rule adds observable
selectivity beyond having two valid, fresh books.

### Key Assumptions

- Grain: one causal end-of-second paired-book state for each of 24 non-overlapping BTC five-minute markets.
- Source: preserved v1 `book` and `chg` events plus the conversion manifest. The resolution manifest,
  terminal directions, BTC reference tapes, and strategy outcomes are never loaded.
- Freshness: both books must be initialized, internally valid, have positive top-three depth, and be no
  older than 30 seconds.
- Timing: native file order is authoritative. Regressing source timestamps are clamped to the latest
  already-seen timestamp for that condition, preventing a late-arriving event from time-travelling into
  a previously sampled second.
- Tick: initialize at `0.01`; persist `0.001` after any causal positive top bid/ask crosses below `0.04`
  or above `0.96`, matching the registered replay policy.
- The registered candidate-time proxy is elapsed seconds `[120, 180)`. Full-window results are reported
  separately. No alternate threshold is searched.


## Data

### 1. Load label-free sources and prove the intended grain


In [1]:
from __future__ import annotations

import gzip
import hashlib
import json
import math
import statistics
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path('/Users/ttoomm/Documents/PolyMomentum')
CAPTURE = Path('/private/tmp/fresh-block-canary-recovered/segment_001')
CONVERTED = CAPTURE / 'converted_v10'
MANIFEST_PATH = CONVERTED / 'manifest.json'
PREVIOUS_DIAGNOSTIC_PATH = ROOT / 'deploy/promotions/evidence/strategy_registry/20260721_paired_book_pressure_redundancy_diagnostic.json'
RUST_SCORER_PATH = ROOT / 'rust_engine/src/strategy_builder.rs'
OUTPUT_PATH = ROOT / 'deploy/promotions/evidence/strategy_registry/20260721_binary_complement_residual_independence_diagnostic.json'

MAX_BOOK_AGE_SECONDS = 30.0
DEPTH_LEVELS = 3
EPSILON = 1e-12

manifest = json.loads(MANIFEST_PATH.read_text())
previous_diagnostic = json.loads(PREVIOUS_DIAGNOSTIC_PATH.read_text())
assert manifest['stats']['skipped_malformed_raw'] == 0
assert manifest['stats']['skipped_unknown_market'] == 0
assert manifest['stats']['skipped_unknown_token'] == 0

event_files = [Path(row['path']) for row in manifest['hours']]
assert all(path.exists() for path in event_files)
markets = manifest['markets']
assert len(markets) == 24

def parse_utc(value: str) -> float:
    return datetime.fromisoformat(value.replace('Z', '+00:00')).timestamp()

market_windows = {}
tokens_by_condition = defaultdict(dict)
condition_by_token = {}
for condition_id, market in markets.items():
    close_ts = parse_utc(market['end_date'])
    market_windows[condition_id] = {'open_ts': close_ts - 300.0, 'close_ts': close_ts}
    for outcome in market['outcomes']:
        name = outcome['name'].lower()
        assert name in {'up', 'down'}
        token_id = outcome['token_id']
        tokens_by_condition[condition_id][name] = token_id
        condition_by_token[token_id] = condition_id
assert all(set(pair) == {'up', 'down'} for pair in tokens_by_condition.values())

ordered_windows = sorted((row['open_ts'], row['close_ts']) for row in market_windows.values())
assert all(left[1] <= right[0] for left, right in zip(ordered_windows, ordered_windows[1:]))
possible_seconds = 24 * 300
assert possible_seconds == 7200

print({
    'conditions': len(markets),
    'possible_one_hz_samples': possible_seconds,
    'event_files': [path.name for path in event_files],
    'resolution_manifest_loaded': False,
    'terminal_labels_loaded': False,
})


{'conditions': 24, 'possible_one_hz_samples': 7200, 'event_files': ['2026-07-15T06.v1.candles.jsonl.gz', '2026-07-15T07.v1.candles.jsonl.gz', '2026-07-15T08.v1.candles.jsonl.gz'], 'resolution_manifest_loaded': False, 'terminal_labels_loaded': False}


### 2. Reconstruct causal paired books in native file order


In [2]:
def fresh_book_state() -> dict:
    return {
        'bids': {}, 'asks': {}, 'best_bid': 0.0, 'best_ask': 0.0,
        'last_update': None, 'has_snapshot': False,
    }

def quantile(values: list[float], probability: float) -> float | None:
    if not values:
        return None
    ordered = sorted(values)
    position = (len(ordered) - 1) * probability
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    weight = position - lower
    return ordered[lower] * (1.0 - weight) + ordered[upper] * weight

def distribution(values: list[float]) -> dict:
    return {
        'count': len(values),
        'min': min(values) if values else None,
        'p50': quantile(values, 0.50),
        'p90': quantile(values, 0.90),
        'p99': quantile(values, 0.99),
        'max': max(values) if values else None,
        'mean': statistics.fmean(values) if values else None,
    }

def pearson(left: list[float], right: list[float]) -> float | None:
    if len(left) != len(right) or len(left) < 2:
        return None
    left_mean = statistics.fmean(left)
    right_mean = statistics.fmean(right)
    numerator = sum((a - left_mean) * (b - right_mean) for a, b in zip(left, right))
    left_ss = sum((a - left_mean) ** 2 for a in left)
    right_ss = sum((b - right_mean) ** 2 for b in right)
    denominator = math.sqrt(left_ss * right_ss)
    return numerator / denominator if denominator > 0 else None

def apply_event(book: dict, event: dict, effective_ts: float) -> None:
    if event['ev'] == 'book':
        book['bids'] = {float(price): float(size) for price, size in event['bids'] if float(size) > 0}
        book['asks'] = {float(price): float(size) for price, size in event['asks'] if float(size) > 0}
        book['has_snapshot'] = True
    elif event['ev'] == 'chg':
        side = 'bids' if event['s'] == 'BUY' else 'asks'
        price = float(event['p'])
        size = float(event['sz'])
        if size > 0:
            book[side][price] = size
        else:
            book[side].pop(price, None)
    else:
        raise AssertionError(f"unexpected event type {event['ev']}")
    book['best_bid'] = float(event['bb'])
    book['best_ask'] = float(event['ba'])
    book['last_update'] = effective_ts

def book_metrics(book: dict, cutoff: float) -> tuple[dict | None, str | None]:
    if not book['has_snapshot'] or book['last_update'] is None:
        return None, 'missing_snapshot'
    age_s = cutoff - book['last_update']
    if age_s < -1e-9:
        return None, 'future_update'
    if age_s > MAX_BOOK_AGE_SECONDS:
        return None, 'stale_book'
    best_bid = book['best_bid']
    best_ask = book['best_ask']
    if not (0.0 < best_bid < best_ask < 1.0):
        return None, 'invalid_top'
    bid_levels = sorted(
        ((price, size) for price, size in book['bids'].items() if size > 0 and price <= best_bid + 1e-9),
        reverse=True,
    )[:DEPTH_LEVELS]
    ask_levels = sorted(
        ((price, size) for price, size in book['asks'].items() if size > 0 and price >= best_ask - 1e-9)
    )[:DEPTH_LEVELS]
    bid_depth = sum(size for _, size in bid_levels)
    ask_depth = sum(size for _, size in ask_levels)
    if bid_depth <= 0 or ask_depth <= 0:
        return None, 'missing_positive_depth'
    midpoint = (best_bid + best_ask) / 2.0
    microprice = (best_ask * bid_depth + best_bid * ask_depth) / (bid_depth + ask_depth)
    return {
        'best_bid': best_bid, 'best_ask': best_ask,
        'bid_depth': bid_depth, 'ask_depth': ask_depth,
        'midpoint': midpoint, 'microprice': microprice,
        'spread': best_ask - best_bid, 'age_s': age_s,
        'last_update': book['last_update'],
    }, None

books = defaultdict(fresh_book_state)
condition_tick = defaultdict(lambda: 0.01)
last_raw_ts = defaultdict(lambda: float('-inf'))
last_effective_ts = defaultdict(lambda: float('-inf'))
next_second = {cid: int(window['open_ts']) for cid, window in market_windows.items()}
native_timestamp_regressions = Counter()
event_counts = Counter()
samples = []
tick_transition_conditions = set()
tick_transition_offsets = []

def capture_second(condition_id: str, second: int) -> None:
    window = market_windows[condition_id]
    elapsed_s = second - int(window['open_ts'])
    cutoff = second + 1.0 - 1e-9
    pair = tokens_by_condition[condition_id]
    up, up_reason = book_metrics(books[pair['up']], cutoff)
    down, down_reason = book_metrics(books[pair['down']], cutoff)
    base = {
        'condition_id': condition_id,
        'second': second,
        'elapsed_s': elapsed_s,
        'candidate_interval': 120 <= elapsed_s < 180,
        'causal_tick': condition_tick[condition_id],
    }
    if up is None or down is None:
        reasons = sorted({reason for reason in (up_reason, down_reason) if reason})
        samples.append({**base, 'valid': False, 'invalid_reasons': reasons})
        return
    mid_residual = up['midpoint'] + down['midpoint'] - 1.0
    micro_residual = up['microprice'] + down['microprice'] - 1.0
    threshold = 2.0 * condition_tick[condition_id]
    mid_pass = abs(mid_residual) <= threshold + EPSILON
    micro_pass = abs(micro_residual) <= threshold + EPSILON
    samples.append({
        **base,
        'valid': True,
        'mid_residual': mid_residual,
        'micro_residual': micro_residual,
        'mid_pass': mid_pass,
        'micro_pass': micro_pass,
        'fixed_pass': mid_pass and micro_pass,
        'threshold': threshold,
        'max_abs_residual': max(abs(mid_residual), abs(micro_residual)),
        'spread_sum': up['spread'] + down['spread'],
        'max_age_s': max(up['age_s'], down['age_s']),
        'age_skew_s': abs(up['age_s'] - down['age_s']),
        'update_skew_s': abs(up['last_update'] - down['last_update']),
        'depth_mirror_abs': max(
            abs(up['bid_depth'] - down['ask_depth']),
            abs(up['ask_depth'] - down['bid_depth']),
        ),
        'cross_touch_abs': max(
            abs(up['best_bid'] + down['best_ask'] - 1.0),
            abs(up['best_ask'] + down['best_bid'] - 1.0),
        ),
    })

for event_path in event_files:
    with gzip.open(event_path, 'rt') as handle:
        for line in handle:
            event = json.loads(line)
            if event['ev'] == 'trade':
                event_counts['trade'] += 1
                continue
            condition_id = event['mkt']
            if condition_id not in markets:
                continue
            raw_ts = float(event['ts'])
            if raw_ts + 1e-9 < last_raw_ts[condition_id]:
                native_timestamp_regressions[condition_id] += 1
            last_raw_ts[condition_id] = max(last_raw_ts[condition_id], raw_ts)
            effective_ts = max(last_effective_ts[condition_id], raw_ts)
            last_effective_ts[condition_id] = effective_ts
            close_second = int(market_windows[condition_id]['close_ts'])
            while next_second[condition_id] < close_second and next_second[condition_id] + 1.0 - 1e-9 < effective_ts:
                capture_second(condition_id, next_second[condition_id])
                next_second[condition_id] += 1
            apply_event(books[event['tok']], event, effective_ts)
            event_counts[event['ev']] += 1
            top_prices = [float(event['bb']), float(event['ba'])]
            crosses = any(price > 0 and (price < 0.04 or price > 0.96) for price in top_prices)
            if crosses and condition_tick[condition_id] > 0.001 + EPSILON:
                condition_tick[condition_id] = 0.001
                tick_transition_conditions.add(condition_id)
                tick_transition_offsets.append(effective_ts - market_windows[condition_id]['open_ts'])

for condition_id, window in market_windows.items():
    close_second = int(window['close_ts'])
    while next_second[condition_id] < close_second:
        capture_second(condition_id, next_second[condition_id])
        next_second[condition_id] += 1

assert len(samples) == possible_seconds
print({
    'events': dict(event_counts),
    'native_timestamp_regressions': sum(native_timestamp_regressions.values()),
    'samples': len(samples),
    'tick_transition_conditions': len(tick_transition_conditions),
})


{'events': {'book': 120475, 'chg': 7561554}, 'native_timestamp_regressions': 865, 'samples': 7200, 'tick_transition_conditions': 24}


## Results

### 3. Test clause independence at the registered threshold


In [3]:
def population_summary(rows: list[dict]) -> dict:
    valid = [row for row in rows if row['valid']]
    invalid = [row for row in rows if not row['valid']]
    fixed_failures = [row for row in valid if not row['fixed_pass']]
    midpoint_failures = [row for row in valid if not row['mid_pass']]
    microprice_failures = [row for row in valid if not row['micro_pass']]
    return {
        'possible_samples': len(rows),
        'valid_pair_samples': len(valid),
        'valid_pair_coverage': len(valid) / len(rows) if rows else None,
        'pair_availability_rejections': len(invalid),
        'fixed_rule_passes': len(valid) - len(fixed_failures),
        'fixed_rule_rejections': len(fixed_failures),
        'fixed_rule_pass_rate_among_valid': (len(valid) - len(fixed_failures)) / len(valid) if valid else None,
        'midpoint_clause_rejections': len(midpoint_failures),
        'microprice_clause_rejections': len(microprice_failures),
        'microprice_incremental_rejections_beyond_midpoint': sum(row['mid_pass'] and not row['micro_pass'] for row in valid),
        'midpoint_incremental_rejections_beyond_microprice': sum(row['micro_pass'] and not row['mid_pass'] for row in valid),
        'conditions_with_any_fixed_rule_rejection': len({row['condition_id'] for row in fixed_failures}),
        'conditions_with_any_pair_unavailability': len({row['condition_id'] for row in invalid}),
        'registered_threshold': distribution([row['threshold'] for row in valid]),
        'max_abs_residual': distribution([row['max_abs_residual'] for row in valid]),
        'minimum_threshold_headroom': min((row['threshold'] - row['max_abs_residual'] for row in valid), default=None),
        'maximum_threshold_utilization': max((row['max_abs_residual'] / row['threshold'] for row in valid), default=None),
    }

full_rows = samples
candidate_rows = [row for row in samples if row['candidate_interval']]
full_summary = population_summary(full_rows)
candidate_summary = population_summary(candidate_rows)
valid_rows = [row for row in samples if row['valid']]
valid_candidate_rows = [row for row in candidate_rows if row['valid']]

assert full_summary['fixed_rule_rejections'] == 0
assert candidate_summary['fixed_rule_rejections'] == 0
assert full_summary['microprice_incremental_rejections_beyond_midpoint'] == 0
assert candidate_summary['microprice_incremental_rejections_beyond_midpoint'] == 0

comparison = {
    'prior_raw_timestamp_sampler': {
        'valid_pair_samples': previous_diagnostic['structural_results']['valid_paired_samples'],
        'abs_midpoint_residual_max': previous_diagnostic['structural_results']['abs_midpoint_sum_residual']['max'],
        'abs_microprice_residual_max': previous_diagnostic['structural_results']['abs_microprice_sum_residual']['max'],
    },
    'monotone_clamped_sampler': {
        'valid_pair_samples': full_summary['valid_pair_samples'],
        'abs_midpoint_residual_max': max(abs(row['mid_residual']) for row in valid_rows),
        'abs_microprice_residual_max': max(abs(row['micro_residual']) for row in valid_rows),
    },
}

print({'full_window': full_summary, 'candidate_interval': candidate_summary})
print({'timestamp_sensitivity': comparison})


{'full_window': {'possible_samples': 7200, 'valid_pair_samples': 6465, 'valid_pair_coverage': 0.8979166666666667, 'pair_availability_rejections': 735, 'fixed_rule_passes': 6465, 'fixed_rule_rejections': 0, 'fixed_rule_pass_rate_among_valid': 1.0, 'midpoint_clause_rejections': 0, 'microprice_clause_rejections': 0, 'microprice_incremental_rejections_beyond_midpoint': 0, 'midpoint_incremental_rejections_beyond_microprice': 0, 'conditions_with_any_fixed_rule_rejection': 0, 'conditions_with_any_pair_unavailability': 24, 'registered_threshold': {'count': 6465, 'min': 0.002, 'p50': 0.02, 'p90': 0.02, 'p99': 0.02, 'max': 0.02, 'mean': 0.01788120649651972}, 'max_abs_residual': {'count': 6465, 'min': 0.0, 'p50': 0.0, 'p90': 2.220446049250313e-16, 'p99': 2.220446049250313e-16, 'max': 0.00022509224978484177, 'mean': 5.876926153143222e-08}, 'minimum_threshold_headroom': 0.001999999999999778, 'maximum_threshold_utilization': 0.011254612489242088}, 'candidate_interval': {'possible_samples': 1440, 'va

### 4. Diagnose whether residual magnitude is spread, staleness, or update-race driven


In [4]:
nonzero_micro = [row for row in valid_rows if abs(row['micro_residual']) > EPSILON]
zero_micro = [row for row in valid_rows if abs(row['micro_residual']) <= EPSILON]
depth_mismatch = [row for row in valid_rows if row['depth_mirror_abs'] > EPSILON]

driver_fields = ['max_age_s', 'age_skew_s', 'update_skew_s', 'spread_sum', 'depth_mirror_abs']
driver_correlations = {
    field: pearson([abs(row['micro_residual']) for row in valid_rows], [row[field] for row in valid_rows])
    for field in driver_fields
}
driver_groups = {
    'zero_microprice_residual': {
        field: distribution([row[field] for row in zero_micro]) for field in driver_fields
    },
    'nonzero_microprice_residual': {
        field: distribution([row[field] for row in nonzero_micro]) for field in driver_fields
    },
}

invalid_reason_counts = Counter()
for row in samples:
    if not row['valid']:
        invalid_reason_counts.update(row['invalid_reasons'])

time_buckets = []
for bucket_start in range(0, 300, 60):
    bucket_rows = [row for row in samples if bucket_start <= row['elapsed_s'] < bucket_start + 60]
    summary = population_summary(bucket_rows)
    time_buckets.append({
        'elapsed_seconds': f'{bucket_start}-{bucket_start + 59}',
        'possible': summary['possible_samples'],
        'valid': summary['valid_pair_samples'],
        'coverage': summary['valid_pair_coverage'],
        'fixed_rule_rejections': summary['fixed_rule_rejections'],
    })

per_condition = []
for condition_id in sorted(markets, key=lambda cid: market_windows[cid]['open_ts']):
    rows = [row for row in samples if row['condition_id'] == condition_id]
    summary = population_summary(rows)
    per_condition.append({
        'condition_id': condition_id,
        'valid_pair_samples': summary['valid_pair_samples'],
        'coverage': summary['valid_pair_coverage'],
        'fixed_rule_rejections': summary['fixed_rule_rejections'],
        'max_abs_residual': summary['max_abs_residual']['max'],
    })

driver_result = {
    'nonzero_microprice_residual_samples': len(nonzero_micro),
    'depth_mismatch_samples': len(depth_mismatch),
    'nonzero_microprice_and_depth_mismatch_overlap': sum(
        abs(row['micro_residual']) > EPSILON and row['depth_mirror_abs'] > EPSILON for row in valid_rows
    ),
    'midpoint_residual_nonzero_samples': sum(abs(row['mid_residual']) > EPSILON for row in valid_rows),
    'cross_touch_error_nonzero_samples': sum(row['cross_touch_abs'] > EPSILON for row in valid_rows),
    'correlations_with_abs_microprice_residual': driver_correlations,
    'group_summaries': driver_groups,
    'invalid_reason_counts': dict(invalid_reason_counts),
    'observed_freshness_rejections': invalid_reason_counts['stale_book'],
    'observed_missing_snapshot_rejections': invalid_reason_counts['missing_snapshot'],
    'coverage_by_market_min': min(row['coverage'] for row in per_condition),
    'coverage_by_market_median': quantile([row['coverage'] for row in per_condition], 0.5),
    'coverage_by_market_max': max(row['coverage'] for row in per_condition),
}

print({'driver_result': driver_result})
print({'coverage_by_minute': time_buckets})


{'driver_result': {'nonzero_microprice_residual_samples': 5, 'depth_mismatch_samples': 5, 'nonzero_microprice_and_depth_mismatch_overlap': 5, 'midpoint_residual_nonzero_samples': 0, 'cross_touch_error_nonzero_samples': 0, 'correlations_with_abs_microprice_residual': {'max_age_s': -0.0016912253302905586, 'age_skew_s': None, 'update_skew_s': None, 'spread_sum': -0.0025198461071533965, 'depth_mirror_abs': 0.7323199843503274}, 'group_summaries': {'zero_microprice_residual': {'max_age_s': {'count': 6460, 'min': 0.0, 'p50': 0.002000093460083008, 'p90': 0.009000062942504883, 'p99': 0.01399993896484375, 'max': 1.8299999237060547, 'mean': 0.005110369156757745}, 'age_skew_s': {'count': 6460, 'min': 0.0, 'p50': 0.0, 'p90': 0.0, 'p99': 0.0, 'max': 0.0, 'mean': 0.0}, 'update_skew_s': {'count': 6460, 'min': 0.0, 'p50': 0.0, 'p90': 0.0, 'p99': 0.0, 'max': 0.0, 'mean': 0.0}, 'spread_sum': {'count': 6460, 'min': 0.002, 'p50': 0.020000000000000018, 'p90': 0.020000000000000018, 'p99': 0.08000000000000003

### 5. Persist the source-pinned diagnostic artifact


In [5]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

source_hashes = {path.name: sha256_file(path) for path in event_files}
source_hashes[MANIFEST_PATH.name] = sha256_file(MANIFEST_PATH)
source_hashes[str(RUST_SCORER_PATH.relative_to(ROOT))] = sha256_file(RUST_SCORER_PATH)

prior_hashes = previous_diagnostic['source_authority']['sha256']
for path in event_files:
    assert source_hashes[path.name] == prior_hashes[path.name]
assert source_hashes[MANIFEST_PATH.name] == prior_hashes[MANIFEST_PATH.name]

all_zero_selectivity = (
    full_summary['fixed_rule_rejections'] == 0
    and candidate_summary['fixed_rule_rejections'] == 0
)
status = (
    'DIAGNOSTIC_ONLY_REGISTERED_RESIDUAL_MAGNITUDE_HAS_ZERO_OBSERVED_SELECTIVITY_ACTIVE_RULE_UNCHANGED'
    if all_zero_selectivity
    else 'DIAGNOSTIC_ONLY_REGISTERED_RESIDUAL_MAGNITUDE_SHOWS_OBSERVED_SELECTIVITY_ACTIVE_RULE_UNCHANGED'
)

evidence = {
    'schema_version': 1,
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'mechanism_id': 'binary_complement_coherence_v1',
    'status': status,
    'decision_question': 'At the registered two-causal-tick threshold, do midpoint and microprice residual magnitudes add observed selectivity beyond paired-top validity?',
    'source_authority': {
        'capture': 'abandoned fresh-block-canary captured 2026-07-15T06:49Z through 2026-07-15T08:50Z',
        'capture_conditions': len(markets),
        'distilled_files': [path.name for path in event_files],
        'sha256': source_hashes,
        'resolution_manifest_loaded': False,
        'terminal_labels_loaded': False,
        'btc_reference_tapes_loaded': False,
        'strategy_outcomes_loaded': False,
    },
    'data_quality': {
        'grain': 'one causal end-of-second paired-book state per condition-second',
        'possible_samples': len(samples),
        'native_timestamp_regressions_observed': sum(native_timestamp_regressions.values()),
        'timestamp_policy': 'preserve native file order; clamp each condition source timestamp to its monotone seen maximum before sampling',
        'skipped_malformed_raw': manifest['stats']['skipped_malformed_raw'],
        'skipped_unknown_market': manifest['stats']['skipped_unknown_market'],
        'skipped_unknown_token': manifest['stats']['skipped_unknown_token'],
        'promotion_or_exact_replay_eligible': False,
        'quality_assessment': 'SHARE_WITH_CAVEATS_FOR_LABEL_FREE_MECHANISM_SCREEN_ONLY',
        'reason': '24-market diagnostic capture with timestamp regressions and incomplete valid paired tops; suitable for observed structural selectivity, not settlement prediction or profitability',
    },
    'methodology': {
        'sampling': 'end-of-second causal state using native event order and monotone-clamped per-condition timestamps',
        'freshness_seconds': MAX_BOOK_AGE_SECONDS,
        'depth_levels': DEPTH_LEVELS,
        'registered_candidate_interval_elapsed_seconds': [120, 180],
        'causal_tick_policy': 'initialize 0.01; persist 0.001 after a positive top bid or ask crosses below 0.04 or above 0.96',
        'registered_rule': 'max(abs(midpoint_sum_residual), abs(microprice_sum_residual)) <= 2 * causal_tick',
        'alternate_thresholds_tested': 0,
    },
    'structural_results': {
        'full_window': full_summary,
        'registered_candidate_interval': candidate_summary,
        'timestamp_policy_sensitivity': comparison,
        'driver_diagnostic': driver_result,
        'coverage_by_minute': time_buckets,
        'per_condition': per_condition,
        'tick_transition_conditions': len(tick_transition_conditions),
        'tick_transition_offsets_seconds': distribution(tick_transition_offsets),
    },
    'mechanism_assessment': {
        'midpoint_residual_clause': 'ZERO_OBSERVED_REJECTIONS_AT_REGISTERED_THRESHOLD',
        'microprice_residual_clause': 'ZERO_OBSERVED_INCREMENTAL_REJECTIONS_AT_REGISTERED_THRESHOLD',
        'paired_top_validity': 'ONLY_OBSERVED_SOURCE_OF_SELECTIVITY_IN_THIS_CAPTURE',
        'interpretation': 'The registered residual magnitude added no observed selectivity beyond paired-top validity. All 735 invalid samples failed the valid-top check; no staleness or missing-snapshot rejection was observed.',
        'active_binary_complement_rule_changed': False,
        'why_not_change_now': 'The forward block is frozen and sealed. This diagnostic is label-free, single-capture, and cannot estimate settlement prediction or economic value.',
    },
    'decision': {
        'strategy_adjustment': 'NO_PARAMETER_OR_RULE_CHANGE_DURING_FROZEN_FORWARD_BLOCK',
        'research_action': 'Do not attribute a future block pass to residual magnitude unless labeled forward evidence shows selectivity beyond paired-top validity; reject residual simplifications or replacements until the sealed decision boundary.',
        'a_plus_claim': False,
        'profitability_claim': False,
        'live_trading': 'OFF',
    },
    'limitations': [
        'Only 24 diagnostic markets from one capture are observed.',
        'The analysis intentionally excludes terminal labels and cannot measure prediction, loss removal, PnL, or causal profitability.',
        'Timestamp clamping prevents time travel but cannot reconstruct an unavailable independent receive timestamp.',
        'One-hertz sampling matches the registered opportunity-report cadence but can miss sub-second transient residuals.',
    ],
}

temporary = OUTPUT_PATH.with_name(f'{OUTPUT_PATH.name}.tmp')
temporary.write_text(json.dumps(evidence, indent=2, sort_keys=True) + '\n')
temporary.replace(OUTPUT_PATH)

print({
    'artifact': str(OUTPUT_PATH.relative_to(ROOT)),
    'status': evidence['status'],
    'full_fixed_rejections': full_summary['fixed_rule_rejections'],
    'candidate_fixed_rejections': candidate_summary['fixed_rule_rejections'],
})


{'artifact': 'deploy/promotions/evidence/strategy_registry/20260721_binary_complement_residual_independence_diagnostic.json', 'status': 'DIAGNOSTIC_ONLY_REGISTERED_RESIDUAL_MAGNITUDE_HAS_ZERO_OBSERVED_SELECTIVITY_ACTIVE_RULE_UNCHANGED', 'full_fixed_rejections': 0, 'candidate_fixed_rejections': 0}


## Takeaways

- This notebook evaluates **observed structural selectivity only**. It cannot establish settlement prediction or profitability.
- The active frozen rule is unchanged. The sealed block remains the only labeled forward decision boundary.
- A future block pass must be interpreted against paired-top validity, because this diagnostic tests whether residual magnitude itself ever changed a decision. No freshness or missing-snapshot rejection was observed in this capture.
